# CIFAR-10 Image Classification — ANN vs CNN

A reproducible comparison of a fully connected neural network (ANN) and a convolutional neural network (CNN) on CIFAR-10.

**Evaluation policy:** the test set is kept completely untouched until final evaluation. Training and model selection use only the training/validation data.

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import datasets, layers, models, callbacks


## 2. Reproducibility

In [ ]:
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)


## 3. Load CIFAR-10

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = datasets.cifar10.load_data()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Training images:", X_train_full.shape)
print("Test images:", X_test.shape)


## 4. Quick Dataset Inspection

In [ ]:
print("Unique classes:", np.unique(y_train_full.ravel()))
print("Class counts:\n", pd.Series(y_train_full.ravel()).value_counts().sort_index())

plt.figure(figsize=(4, 4))
plt.imshow(X_train_full[0])
plt.title(class_names[int(y_train_full[0])])
plt.axis("off")
plt.show()


## 5. Preprocessing and Train/Validation Split

In [ ]:
# Normalize pixels to [0, 1].
X_train_full = X_train_full.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

y_train_full = y_train_full.ravel().astype("int32")
y_test = y_test.ravel().astype("int32")

# Split only the original training set. The test set remains untouched.
train_idx, val_idx = train_test_split(
    np.arange(len(X_train_full)),
    test_size=0.10,
    random_state=SEED,
    stratify=y_train_full,
)

X_train = X_train_full[train_idx]
y_train = y_train_full[train_idx]
X_val = X_train_full[val_idx]
y_val = y_train_full[val_idx]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


## 6. ANN Baseline

In [ ]:
ann = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Flatten(),
    layers.Dense(1024, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(10, activation="softmax"),
])

ann.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

ann.summary()


In [ ]:
ann_history = ann.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=128,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)
    ],
    verbose=1,
)


In [ ]:
ann_test_loss, ann_test_acc = ann.evaluate(X_test, y_test, verbose=0)
print(f"ANN test accuracy: {ann_test_acc:.4f}")


## 7. CNN with Data Augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
], name="data_augmentation")

cnn = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,

    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.35),
    layers.Dense(10, activation="softmax"),
])

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cnn.summary()


In [ ]:
cnn_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    ),
    callbacks.ModelCheckpoint(
        "cnn_cifar10_best.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
    ),
]

cnn_history = cnn.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=128,
    callbacks=cnn_callbacks,
    verbose=1,
)


## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ann_history.history["accuracy"], label="ANN train")
axes[0].plot(ann_history.history["val_accuracy"], label="ANN val")
axes[0].plot(cnn_history.history["accuracy"], label="CNN train")
axes[0].plot(cnn_history.history["val_accuracy"], label="CNN val")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(ann_history.history["loss"], label="ANN train")
axes[1].plot(ann_history.history["val_loss"], label="ANN val")
axes[1].plot(cnn_history.history["loss"], label="CNN train")
axes[1].plot(cnn_history.history["val_loss"], label="CNN val")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


## 9. Final Evaluation on the Untouched Test Set

In [ ]:
cnn_test_loss, cnn_test_acc = cnn.evaluate(X_test, y_test, verbose=0)
print(f"CNN test accuracy: {cnn_test_acc:.4f}")


In [ ]:
y_prob = cnn.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_test, y_pred, target_names=class_names, digits=4))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(10), class_names, rotation=45, ha="right")
plt.yticks(range(10), class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("CNN Confusion Matrix")
plt.tight_layout()
plt.show()


## 10. Sample Predictions

In [ ]:
sample_idx = np.random.default_rng(SEED).choice(len(X_test), size=12, replace=False)
sample_probs = cnn.predict(X_test[sample_idx], verbose=0)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, idx, probs in zip(axes.ravel(), sample_idx, sample_probs):
    pred = int(np.argmax(probs))
    true = int(y_test[idx])
    title = f"True: {class_names[true]}\\nPred: {class_names[pred]}"
    ax.imshow(X_test[idx])
    ax.set_title(title, color="green" if pred == true else "red")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 11. Save the Final CNN

In [ ]:
cnn.save("cnn_cifar10_final.keras")
print("Saved: cnn_cifar10_final.keras")


## Evaluation Notes

- The 50,000-image training split is divided into training and validation subsets.
- The 10,000-image CIFAR-10 test set is never used for fitting or model selection.
- The CNN uses augmentation, batch normalization, dropout, early stopping, and learning-rate reduction.
- Final metrics must be regenerated by running the notebook from top to bottom after code changes.
- Do not reuse historical metrics from the previous leakage-affected notebook.